# DATATHON 2026 HS Track - Task 1: MarketSense
## Multimodal Price Class Prediction for Rentals

**Goal:** predict the price class (`target`, 0–4) of a rental listing from tabular, text, and location signals.

**Approach (validated locally with 5-fold CV):**
1. The categorical columns `city` and `room_type` are *intentionally corrupted* (random casing, typos like `Entir3`/`Entyre`, separators, Indonesian variants). We normalize them with regex rules.
2. Only ~845 unique lat/lon pairs exist in 99k rows → location is a **cluster id**, the single strongest feature (`latlon_key`).
3. ~70% of test `host_id`s appear in train → host-level aggregates are informative and leak-free (computed without the target).
4. Model: **XGBoost** (hist, native categorical) with StratifiedKFold 5-fold; test prediction = average of fold probabilities.

| Version | Model | CV Acc | CV Macro-F1 | Notes |
|---|---|---|---|---|
| v1 | XGBoost, 94 features | **0.6173** | 0.5747 | this notebook |
| v2 | + TF-IDF/SVD text | 0.607 (fold 0) | - | text hurt → dropped |

## 1. Imports

In [ ]:
import ast
import os
import re
import time

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

## 2. Configuration

In [ ]:
SEED = 42
N_FOLDS = 5
N_CLASS = 5

# Auto-detect Kaggle vs local execution
def find_data_dir() -> str:
    root = '/kaggle/input'
    if os.path.isdir(root):
        for d, _, files in os.walk(root):
            if 'train.csv' in files and 'test.csv' in files:
                return d
    return '.'  # local: files next to the notebook

DATA_DIR = find_data_dir()
print('data dir:', DATA_DIR)

XGB_PARAMS = dict(
    objective='multi:softprob', num_class=N_CLASS, eval_metric='mlogloss',
    learning_rate=0.05, max_depth=8, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method='hist', seed=SEED, n_jobs=-1,
)

## 3. Load Dataset

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
y = train['target'].values

print('train:', train.shape, '| test:', test.shape)
train['target'].value_counts().sort_index()

## 4. Exploratory Data Analysis

Key findings that drive every later decision:
- **Target imbalance:** class 2 ≈ 39%, classes 0 and 4 ≈ 6–7% → use *Stratified*KFold and report macro-F1 next to accuracy.
- **`city` / `room_type` are noisy on purpose** (`CITY-C`, `cty_c`, `ID_City_C_01`, `Entir3 home`, `apartemen`, `pRiVaTe RoOm`, …).
- **`calendar_updated` is 100% missing** → dropped.
- **~845 unique coordinates** for 99,000 rows → listings sit on a grid of location clusters.
- Higher price classes (3–4) have *fewer reviews* and *lower estimated occupancy* expensive places get booked less.

In [ ]:
print('missing % (top 12):')
print((train.isna().mean() * 100).round(1).sort_values(ascending=False).head(12))
print()
print('unique lat/lon pairs:', train.groupby(['latitude', 'longitude']).ngroups)
print('noisy city examples:', train['city'].sample(5, random_state=SEED).tolist())
print()
print('mean occupancy / reviews per class:')
print(train.groupby('target')[['estimated_occupancy_l365d', 'number_of_reviews']].mean().round(1))

## 5. Data Cleaning

Regex normalizers for the corrupted categoricals. Leet-speak (`3`→`e`) is undone first, then keyword matching maps every variant to a canonical value. Verified on train: 0 rows end up in an unmapped bucket for `room_type`; every `city` string contains the city letter and is recovered by the pattern.

In [ ]:
def normalize_city(s: str) -> str:
    """'CITY-C', 'cty_c', 'ID_City_B_01', 'City_C_M3tropolitan' -> 'c', 'b', ..."""
    s = str(s).lower().replace('3', 'e')
    m = re.search(r'(?:city|cty)[^a-z]*([abcd])(?![a-z])', s)
    return m.group(1) if m else 'unk'


def normalize_room_type(s: str) -> str:
    s = str(s).lower().replace('3', 'e')
    s = re.sub(r'[^a-z]', ' ', s)
    if any(k in s for k in ('entire', 'entyre', 'entir', 'whole', 'full house',
                            'apartemen', 'residential')):
        return 'entire'
    if any(k in s for k in ('privat', 'privte', 'own room')):
        return 'private'
    if any(k in s for k in ('shar', 'shre', 'kamar')):
        return 'shared'
    if any(k in s for k in ('hotel', 'hotell', 'hoteel', 'htl', 'boutique')):
        return 'hotel'
    return 'other'


def parse_percent(s) -> float:
    if pd.isna(s):
        return np.nan
    m = re.search(r'(\d+)', str(s))
    return float(m.group(1)) if m else np.nan


def parse_bathrooms_text(s):
    """'1.5 shared baths' -> (1.5, shared=1, private=0). 'Half-bath' -> 0.5."""
    if pd.isna(s):
        return np.nan, 0, 0
    s = str(s).lower()
    shared = int('shared' in s)
    private = int('private' in s)
    m = re.search(r'(\d+(?:\.\d+)?)', s)
    n = float(m.group(1)) if m else (0.5 if 'half' in s else np.nan)
    return n, shared, private


# sanity check on train
print(train['city'].map(normalize_city).value_counts().to_dict())
print(train['room_type'].map(normalize_room_type).value_counts().to_dict())

## 6. Feature Engineering

Built on train+test combined (**no target used** → no leakage):
- normalized categoricals + `latlon_key` (rounded coordinate pair = location cluster)
- numeric passthrough, percent parsing, boolean flags, `has_license`
- date deltas relative to `date_obtained` (host tenure, review recency)
- amenities: count + 24 keyword flags (pool, hot tub, dishwasher… premium markers)
- text *lengths* only (word counts); TF-IDF/SVD was tested and hurt CV
- ratios (beds per person, availability ratio) and host / location aggregates

In [ ]:
KEY_AMENITIES = [
    'wifi', 'kitchen', 'air conditioning', 'pool', 'free parking', 'gym',
    'elevator', 'washer', 'dryer', 'dishwasher', 'bathtub', 'balcony',
    'heating', 'tv', 'coffee', 'hot tub', 'bbq', 'crib', 'workspace',
    'self check-in', 'lockbox', 'breakfast', 'long term stays', 'pets allowed',
]


def count_words(s) -> int:
    return 0 if pd.isna(s) else len(str(s).split())


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Pure per-row transforms (no target usage)."""
    out = pd.DataFrame(index=df.index)

    # normalized categoricals
    out['city'] = df['city'].map(normalize_city)
    out['room_type'] = df['room_type'].map(normalize_room_type)
    out['property_type'] = (df['property_type'].astype(str).str.lower()
                            .str.replace(r'[^a-z ]', ' ', regex=True)
                            .str.replace(r'\s+', ' ', regex=True).str.strip())
    out['neighbourhood'] = df['neighbourhood'].fillna('missing')
    out['host_response_time'] = df['host_response_time'].fillna('missing')

    # numeric passthrough
    num_cols = ['accommodates', 'bathrooms', 'bedrooms', 'beds',
                'minimum_nights', 'maximum_nights',
                'availability_30', 'availability_60', 'availability_90',
                'availability_365', 'number_of_reviews',
                'estimated_occupancy_l365d', 'review_scores_rating',
                'review_scores_accuracy', 'review_scores_cleanliness',
                'review_scores_communication', 'reviews_per_month',
                'host_total_listings_count', 'latitude', 'longitude']
    for c in num_cols:
        out[c] = pd.to_numeric(df[c], errors='coerce')

    # percentages
    out['host_response_rate'] = df['host_response_rate'].map(parse_percent)
    out['host_acceptance_rate'] = df['host_acceptance_rate'].map(parse_percent)

    # booleans
    for c in ['host_has_profile_pic', 'host_identity_verified',
              'has_availability', 'instant_bookable']:
        out[c] = (df[c] == 't').astype(int)
    out['has_license'] = df['license'].notna().astype(int)

    # bathrooms_text
    bt = df['bathrooms_text'].map(parse_bathrooms_text)
    out['bath_n'] = [t[0] for t in bt]
    out['bath_shared'] = [t[1] for t in bt]
    out['bath_private'] = [t[2] for t in bt]
    out['bathrooms'] = out['bathrooms'].fillna(out['bath_n'])

    # date deltas
    ref = pd.to_datetime(df['date_obtained'], errors='coerce')
    for c in ['host_since', 'first_review', 'last_review']:
        d = pd.to_datetime(df[c], errors='coerce')
        out[f'days_{c}'] = (ref - d).dt.days

    # amenities
    def parse_amen(s):
        try:
            return [a.lower() for a in ast.literal_eval(str(s))]
        except (ValueError, SyntaxError):
            return []
    amen = df['amenities'].map(parse_amen)
    out['n_amenities'] = amen.map(len)
    for k in KEY_AMENITIES:
        out[f'am_{k.replace(" ", "_")}'] = amen.map(
            lambda lst, k=k: int(any(k in a for a in lst)))

    # host_verifications
    hv = df['host_verifications'].fillna('[]').astype(str)
    out['n_verifications'] = hv.str.count(',') + hv.str.contains(r'\w').astype(int)
    out['verif_email'] = hv.str.contains('email').astype(int)
    out['verif_phone'] = hv.str.contains('phone').astype(int)
    out['verif_work_email'] = hv.str.contains('work_email').astype(int)

    # text lengths (word counts only — raw TF-IDF hurt CV)
    for c in ['name', 'description', 'neighborhood_overview', 'host_about',
              'lattest comment']:
        out[f'len_{c.replace(" ", "_")}'] = df[c].map(count_words)

    # ratios / interactions
    out['beds_per_person'] = out['beds'] / out['accommodates'].clip(lower=1)
    out['baths_per_person'] = out['bathrooms'] / out['accommodates'].clip(lower=1)
    out['beds_per_bedroom'] = out['beds'] / out['bedrooms'].clip(lower=1)
    out['occ_x_reviews'] = (out['estimated_occupancy_l365d']
                            * np.log1p(out['number_of_reviews']))
    out['avail_ratio_30_365'] = out['availability_30'] / (out['availability_365'] + 1)
    out['review_span_days'] = out['days_first_review'] - out['days_last_review']
    out['reviews_per_day_active'] = (out['number_of_reviews']
                                     / out['review_span_days'].clip(lower=1))
    out['min_nights_log'] = np.log1p(out['minimum_nights'])
    out['max_nights_log'] = np.log1p(out['maximum_nights'].clip(upper=10000))

    # location cluster id (~845 unique coordinates in the whole dataset)
    out['latlon_key'] = (out['latitude'].round(3).astype(str) + '_'
                         + out['longitude'].round(3).astype(str))
    return out


def add_group_features(all_df: pd.DataFrame, raw_all: pd.DataFrame) -> pd.DataFrame:
    """Aggregates over train+test combined (no target involved -> no leakage)."""
    out = all_df.copy()

    host = raw_all.groupby('host_id').agg(
        host_n_listings=('id', 'count'),
        host_mean_reviews=('number_of_reviews', 'mean'),
        host_mean_occ=('estimated_occupancy_l365d', 'mean'),
        host_mean_accom=('accommodates', 'mean'),
    )
    out = out.join(host, on=raw_all['host_id'])
    out['host_id_freq'] = out['host_n_listings']

    grp = out.groupby('latlon_key')
    out['loc_count'] = grp['accommodates'].transform('count')
    out['loc_mean_accom'] = grp['accommodates'].transform('mean')
    out['loc_mean_occ'] = grp['estimated_occupancy_l365d'].transform('mean')
    out['loc_mean_reviews'] = grp['number_of_reviews'].transform('mean')

    out['neigh_freq'] = out.groupby('neighbourhood')['accommodates'].transform('count')
    out['prop_freq'] = out.groupby('property_type')['accommodates'].transform('count')
    return out


t0 = time.time()
raw_all = pd.concat([train.drop(columns=['target']), test], ignore_index=True)
feats = build_features(raw_all)
feats = add_group_features(feats, raw_all)

CAT_COLS = ['city', 'room_type', 'property_type', 'neighbourhood',
            'host_response_time', 'latlon_key']
for c in CAT_COLS:
    feats[c] = feats[c].astype('category')

X = feats.iloc[:len(train)].reset_index(drop=True)
X_test = feats.iloc[len(train):].reset_index(drop=True)
print(f'{X.shape[1]} features, prep {time.time()-t0:.0f}s')

## 7. Train / Validation Split

**StratifiedKFold (5 folds)** preserves the 6/21/39/27/7% class mix in every fold. Every row is used for validation exactly once (out-of-fold), so the CV score is an unbiased estimate and no data is wasted.

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))
for i, (itr, iva) in enumerate(folds):
    print(f'fold {i}: train={len(itr)} valid={len(iva)}')

## 8. Model Training

XGBoost `hist` with native categorical support. Early stopping on each fold's validation set (100 rounds patience) picks the tree count automatically no manual tuning of `n_estimators`.

In [ ]:
t0 = time.time()
oof = np.zeros((len(X), N_CLASS))
pred = np.zeros((len(X_test), N_CLASS))
dtest = xgb.DMatrix(X_test, enable_categorical=True)

for fold, (itr, iva) in enumerate(folds):
    dtr = xgb.DMatrix(X.iloc[itr], y[itr], enable_categorical=True)
    dva = xgb.DMatrix(X.iloc[iva], y[iva], enable_categorical=True)
    model = xgb.train(XGB_PARAMS, dtr, num_boost_round=3000,
                      evals=[(dva, 'va')], early_stopping_rounds=100,
                      verbose_eval=False)
    best = model.best_iteration + 1
    oof[iva] = model.predict(dva, iteration_range=(0, best))
    pred += model.predict(dtest, iteration_range=(0, best)) / N_FOLDS
    print(f'fold {fold}: acc={accuracy_score(y[iva], oof[iva].argmax(1)):.4f} '
          f'best_iter={model.best_iteration} ({time.time()-t0:.0f}s)')

## 9. Cross Validation

Out-of-fold metrics, this is the number we trust, *not* the public LB (only ~50% of test).

In [ ]:
oof_lbl = oof.argmax(1)
print(f'OOF accuracy   : {accuracy_score(y, oof_lbl):.4f}')
print(f'OOF macro-F1   : {f1_score(y, oof_lbl, average="macro"):.4f}')
print(f'OOF weighted-F1: {f1_score(y, oof_lbl, average="weighted"):.4f}')
print()
print('confusion matrix (rows = true class):')
print(confusion_matrix(y, oof_lbl))
print()
imp = pd.Series(model.get_score(importance_type='gain')).sort_values(ascending=False)
print('top 15 features by gain:')
print(imp.head(15).round(1))

## 10. Prediction

Test prediction = mean of the 5 fold models' probabilities → argmax. Averaging probabilities is a free mini-ensemble: it reduces variance versus training one model on all data.

In [ ]:
test_lbl = pred.argmax(1)
print('predicted class distribution (test):')
print(pd.Series(test_lbl).value_counts(normalize=True).sort_index().round(3))
print('train distribution for comparison:')
print(pd.Series(y).value_counts(normalize=True).sort_index().round(3))

## 11. Submission

In [ ]:
submission = pd.DataFrame({'id': test['id'], 'target': test_lbl})
submission.to_csv('submission.csv', index=False)
print(submission.shape)
submission.head()